### Zigbang 원룸 매물 데이터 수집

In [3]:
import requests
import pandas as pd

#### Process
    - 동이름으로 위도 경도 구하기
    - 위도 경도로 geohash 알아내기
    - geohash로 매물 아이디 가져오기
    - 매물 아이디로 매물 정보 가져오기

#### 1. 동이름으로 위도 경도 구하기

In [13]:
addr = '송강동'
url = f'https://apis.zigbang.com/v2/search?leaseYn=N&q={addr}&serviceType=원룸'
response = requests.get(url)
data = response.json()['items'][0]
lat, lng = data['lat'], data['lng']
lat, lng

(36.43072891235352, 127.38162994384766)

#### 2. 위도 경도로 geohash 알아내기

In [15]:
!pip install geohash2

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for geohash2: filename=geohash2-1.1-py3-none-any.whl size=15556 sha256=e24d3d9070295adcc87f02f18c4f3b3b7a68d7ee3aca6c5b43c2c0f7c563fade
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\00\d5\b6\3fbe4088f7912982f596eaddfd593d16096468a2f13e470ae7
Successfully built geohash2


In [17]:
import geohash2

In [19]:
geohash = geohash2.encode(lat, lng, precision=5)
geohash

'wy6x6'

#### 3. geohash로 매물 아이디 가져오기

In [25]:
url = f'https://apis.zigbang.com/v2/items/oneroom?geohash={geohash}&depositMin=0&rentMin=0&salesTypes[0]=전세&salesTypes[1]=월세&domain=zigbang&checkAnyItemWitho'
response = requests.get(url)
response

<Response [200]>

In [41]:
item_ids = []
for data in response.json()['items']:
    item_ids.append(data['itemId'])

# item_ids = [data['itemId'] for data in response.json()['items']]

len(item_ids), item_ids[:10]

(69,
 [42218916,
  42130598,
  42169826,
  42166871,
  41132294,
  42188772,
  42218879,
  42142106,
  42141950,
  42219743])

#### 4. 매물 아이디로 매물 정보 가져오기

In [43]:
url = 'https://apis.zigbang.com/v2/items/list'
params = {
    'domain': 'zigbang',
    'item_ids': item_ids
}
response = requests.post(url, params)
response

<Response [200]>

In [51]:
pd.options.display.max_columns = 40

In [59]:
df.columns

Index(['item_id', 'section_type', 'images_thumbnail', 'sales_type',
       'sales_title', 'deposit', 'rent', 'size_m2', '공급면적', '전용면적', '계약면적',
       'room_type_title', 'floor', 'floor_string', 'building_floor', 'title',
       'is_first_movein', 'room_type', 'status', 'tags', 'service_type',
       'random_location', 'location', 'manage_cost', 'reg_date', 'is_new',
       'addressOrigin', 'action', 'contract', 'address', 'is_zzim', 'address1',
       'address2', 'address3', 'item_bm_type', 'isCleanLessor', 'zikim'],
      dtype='object')

In [65]:
data = response.json()['items']
df = pd.DataFrame(data)
df = df[['item_id', 'sales_type', 'deposit', 'rent', 'floor', 'building_floor', 'title', 'floor', 'size_m2', 'address1']]
df.head()

,item_id,sales_type,deposit,rent,floor,building_floor,title,floor,size_m2,address1
0,42218916,월세,500,40,1,4,ㅡ다산 ㅡ 리모델링 풀옵션 원룸,1,27.00,대전시 유성구 송강동
1,42130598,월세,200,25,1,3,ㅡ한솔랜드ㅡ단기 가능한 실속있는 준신축풀옵션원룸,1,19.83,대전시 유성구 송강동
2,42169826,월세,300,30,1,3,ㅡ한솔랜드ㅡ최상의 주방분리형의 신축풀옵션원룸ㅡ통 베란다,1,23.14,대전시 유성구 송강동
3,42166871,월세,300,30,1,3,ㅡ다산 ㅡ 베란다넓은 신축분리형풀옵션원룸,1,19.83,대전시 유성구 송강동
4,41132294,월세,200,29,3,3,관평동 가까운 가격저렴한 원룸,3,23.14,대전시 유성구 송강동


In [99]:
# function
def oneroom(addr):
    url = f'https://apis.zigbang.com/v2/search?leaseYn=N&q={addr}&serviceType=원룸'
    response = requests.get(url)
    data = response.json()['items'][0]
    lat, lng = data['lat'], data['lng']
    geohash = geohash2.encode(lat, lng, precision=5)

    url = f'https://apis.zigbang.com/v2/items/oneroom?geohash={geohash}&depositMin=0&rentMin=0&salesTypes[0]=전세&salesTypes[1]=월세&domain=zigbang&checkAnyItemWitho'
    response = requests.get(url)
    item_ids = []
    for data in response.json()['items']:
        item_ids.append(data['itemId'])

    url = 'https://apis.zigbang.com/v2/items/list'
    params = {
        'domain': 'zigbang',
        'item_ids': item_ids
    }
    response = requests.post(url, params)
    data = response.json()['items']
    df = pd.DataFrame(data)
    df = df[df['address1'].str.contains(addr)].reset_index(drop=True)
    df = df[['item_id', 'sales_type', 'deposit', 'rent', 'floor', 'building_floor', 'title', 'floor', 'size_m2', 'address1']]

    return df

In [101]:
df = oneroom('송강동')
df

,item_id,sales_type,deposit,rent,floor,building_floor,title,floor,size_m2,address1
0,42218916,월세,500,40,1,4,ㅡ다산 ㅡ 리모델링 풀옵션 원룸,1,27.0000,대전시 유성구 송강동
1,42130598,월세,200,25,1,3,ㅡ한솔랜드ㅡ단기 가능한 실속있는 준신축풀옵션원룸,1,19.8300,대전시 유성구 송강동
2,42169826,월세,300,30,1,3,ㅡ한솔랜드ㅡ최상의 주방분리형의 신축풀옵션원룸ㅡ통 베란다,1,23.1400,대전시 유성구 송강동
3,42166871,월세,300,30,1,3,ㅡ다산 ㅡ 베란다넓은 신축분리형풀옵션원룸,1,19.8300,대전시 유성구 송강동
4,41132294,월세,200,29,3,3,관평동 가까운 가격저렴한 원룸,3,23.1400,대전시 유성구 송강동
5,42188772,월세,200,25,2,3,ㅡ한솔랜드ㅡ단기 가능한 포근하고 느낌좋은 분리형 풀옵션원룸,2,19.8300,대전시 유성구 송강동
6,42218879,월세,200,25,1,3,"ㅡ다산 ㅡ 단기가능,풀옵션원룸 생활이편리해요",1,16.5289,대전시 유성구 송강동
7,42142106,월세,300,32,3,3,ㅡ다산 ㅡ 리모델링 분리형원룸 풀옵션원룸,3,26.3500,대전시 유성구 송강동
8,42141950,월세,200,24,2,3,ㅡ다산 ㅡ 가성비좋은 풀옵션원룸,2,26.4500,대전시 유성구 송강동
9,42219743,월세,200,27,3,3,"송강동, 리모델링 내부 이쁘게 꾸며놓은집",3,26.4500,대전시 유성구 송강동
